# AIC2026 — OCR L25 trên Colab T4 (GPU)

Chạy PaddleOCR (PP-OCRv6 *medium*) trên GPU T4 để OCR keyframe L25 nhanh 10–20x so với CPU.

### Bước chuẩn bị (làm 1 lần)
1. Tải repo lên **Google Drive** (vd `/content/drive/MyDrive/aic2026`) hoặc `git clone` vào Drive.
2. Copy data vào Drive:
   - `data/raw/Keyframes/...` (chứa thư mục `Keyframes_L25/keyframes/L25_Vxxx/*.jpg`)
   - `data/processed/official_manifest.jsonl`
3. Dùng runtime **GPU** (Menu → Runtime → Change runtime type → T4).

> Output ghi vào **Drive** (`PROJECT_DIR/data/processed/...`) nên reconnect/disconnect không mất kết quả. Luôn có `--resume` → chạy lại lệnh là tiếp tục.


In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Đường dẫn project trong Drive (sửa cho khớp với chỗ bạn lưu)
PROJECT_DIR = '/content/drive/MyDrive/aic2026'
import os
assert os.path.isdir(PROJECT_DIR), f'Không tìm thấy {PROJECT_DIR}. Sửa PROJECT_DIR.'
assert os.path.isdir(f'{PROJECT_DIR}/data/raw/Keyframes/Keyframes_L25'), 'Thiếu data/raw/Keyframes trong Drive.'
assert os.path.isfile(f'{PROJECT_DIR}/data/processed/official_manifest.jsonl'), 'Thiếu official_manifest.jsonl.'
print('OK project at', PROJECT_DIR)

In [ ]:
# 2) Cài paddle bản GPU (CUDA 12.x trên Colab T4)
# Bản CPU cài sẵn trong requirements KHÔNG có CUDA -> phải cài đè bản gpu.
!pip install -q paddlepaddle-gpu==3.2.0 -f https://www.paddlepaddle.org.cn/whl/linux/cuda12.0/
!pip install -q paddleocr==3.7.0

# Cài package (chỉ cần phần OCR: typer + paddleocr)
%cd {PROJECT_DIR}
!pip install -q -e .

# Verify CUDA thực sự dùng được
import paddle
print('is_compiled_with_cuda:', paddle.is_compiled_with_cuda())
assert paddle.is_compiled_with_cuda(), 'Paddle GPU chưa sẵn sàng — kiểm tra runtime có bật GPU không.'

In [ ]:
# 3) Chạy OCR L25 trên GPU (có --resume, ghi output vào Drive)
# batch-size GPU chịu được 32–64 (tăng để nhanh hơn). 
!python -m aic2026.cli ocr-manifest \
    --manifest data/processed/official_manifest.jsonl \
    --output data/processed/official_manifest_ocr_vi.jsonl \
    --keyframes-root data/raw/Keyframes \
    --batch-size 32 --lang vi --model-size medium \
    --device gpu --video-prefix L25 --resume

In [ ]:
# 4) (Tùy chọn) Theo dõi tiến độ — mở cell này chạy song song trong 1 tab khác
# Output nằm trên Drive nên an toàn khi disconnect.
import time, json
OUT = f'{PROJECT_DIR}/data/processed/official_manifest_ocr_vi.jsonl'
def progress():
    done = 0; lines = 0
    with open(OUT) as f:
        for ln in f:
            lines += 1
            if '"ocr_done": true' in ln: done += 1
    return done, lines
while True:
    d, l = progress()
    print(f'L25 OCR done: {d:,} / 37,445 | total lines: {l:,}', flush=True)
    time.sleep(30)

## Lưu ý khi bị disconnect
- File output nằm trên **Drive** → không mất.
- Reconnect → chạy lại **Cell 2 + Cell 3**. `--resume` tự bỏ qua phần đã OCR xong (dựa vào flag `ocr_done` trong output cũ).
- Không chạy 2 cell 3 đồng thời (cùng ghi 1 file → hỏng).
